# Ball Mapper analysis: ball profiles, breadth and depth

## Purpose
Interpret the ε = 3.0 Ball Mapper output by building the ball-level tables
used in the report. Also tests whether creative *breadth* (number of
specialised sub-sectors) or *depth* (strongest single specialisation)
is more closely linked to local socio-economic outcomes.

## Input
- `ball_profiles_census.csv` (ball-level sector profiles with census means)
- `data/processed/nesta_ball.csv`
- `data/processed/census_filtered.csv`

## Main steps
- Build ball-level mean (SD) tables for the census variables
- Extract report tables for selected principal-component and disconnected balls
  (dominant sector and LQ, qualifications, self-employment, house prices)
- Define breadth (number of sector LQs > 1) and depth (maximum sector LQ)
- Ball level (16 principal-component balls): correlate breadth and depth with
  Level 4+ qualifications (Pearson and Spearman), and fit a two-predictor
  linear regression
- TTWA level (173 TTWAs): OLS regressions with HC3 robust standard errors
  of six outcomes on breadth and depth (Level 4+, no qualifications,
  economic inactivity, self-employment, higher managerial, routine)
- Identify illustrative TTWA pairs: narrow/deep places (breadth 1–2,
  above-median depth) against broad places (breadth ≥ 3) with lower depth

## Output
- Ball-level report tables (census mean (SD), selected ball profiles)
- Breadth-depth regression table for the ERP
- Candidate narrow-vs-broad TTWA case-study pairs
- All displayed in the notebook, not saved to file

In [1]:
import pandas as pd

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
profiles = pd.read_csv("/Users/scandimimi/Documents/Manchester/ERP/ball_profiles_full.csv")

In [3]:
profiles.head()

,ball,component_status,is_isolate,n_ttwas,dominant_sector,dominant_LQ,second_sector,second_LQ,dominance_gap,Advertising_LQ,...,pct_routine_mean,pct_no_quals_mean,pct_level4_plus_mean,pct_self_employed_mean,pct_econ_inactive_mean,pop_density_mean,log_pop_density_mean,urban_rural_score_mean,pct_urban_mean,median_house_price_mean
0,0,Principal component,False,70,Architecture,0.75,Design,0.67,0.09,0.34,...,12.59,25.07,23.55,15.87,31.80,3.54,1.22,3.45,58.98,150090.14
1,1,Principal component,False,9,Publishing,2.01,Design,1.07,0.93,0.48,...,11.69,23.29,26.13,15.31,31.17,3.22,1.29,3.37,62.55,177105.61
2,2,Principal component,False,84,Design,0.86,Architecture,0.80,0.06,0.45,...,11.98,23.90,24.70,15.23,30.52,4.61,1.47,3.29,67.00,159256.86
3,3,Principal component,False,3,IT_Software,1.53,Design,1.47,0.06,1.02,...,9.57,18.77,31.42,15.20,26.41,5.02,1.67,3.40,74.57,216500.00
4,4,Principal component,False,9,Film_TV,0.94,Design,0.83,0.11,0.29,...,12.56,24.65,25.04,15.77,31.91,3.29,1.20,3.64,52.07,148779.06


In [4]:
ball_census_stats = pd.read_csv(
    "/Users/scandimimi/Documents/Manchester/ERP/"
    "ball_census_stats_epsilon_3.csv"
)

mean_sd_table = ball_census_stats[
    ["ball", "variable", "n_ttwas", "mean", "std"]
].rename(columns={
    "n_ttwas": "n",
    "std": "sd"
}).sort_values(
    ["ball", "variable"]
).reset_index(drop=True)

mean_sd_table

,ball,variable,n,mean,sd
0,0,log_pop_density,70,1.22,0.73
1,0,median_house_price,70,150090.14,31379.76
2,0,pct_econ_inactive,70,31.80,3.26
3,0,pct_higher_managerial,70,8.25,1.64
4,0,pct_level4_plus,70,23.55,3.59
...,...,...,...,...,...
385,29,pct_routine,1,7.32,NaN
386,29,pct_self_employed,1,19.79,NaN
387,29,pct_urban,1,54.50,NaN
388,29,pop_density,1,2.66,NaN


In [5]:
ball_census_stats["mean_sd"] = (
    ball_census_stats["mean"].map("{:.2f}".format)
    + " ("
    + ball_census_stats["std"].map("{:.2f}".format)
    + ")"
)

mean_sd_wide = (
    ball_census_stats
    .pivot(
        index="ball",
        columns="variable",
        values="mean_sd"
    )
    .reset_index()
)

mean_sd_wide

variable,ball,log_pop_density,median_house_price,pct_econ_inactive,pct_higher_managerial,pct_level4_plus,pct_no_quals,pct_owned,pct_private_rented,pct_routine,pct_self_employed,pct_urban,pop_density,urban_rural_score
0,0,1.22 (0.73),150090.14 (31379.76),31.80 (3.26),8.25 (1.64),23.55 (3.59),25.07 (3.53),67.54 (3.38),15.38 (2.48),12.59 (2.13),15.87 (5.05),58.98 (26.68),3.54 (4.14),3.45 (0.81)
1,1,1.29 (0.55),177105.61 (51028.48),31.17 (5.58),10.57 (3.48),26.13 (5.37),23.29 (5.47),68.31 (4.27),14.74 (2.09),11.69 (2.70),15.31 (4.21),62.55 (19.09),3.22 (2.87),3.37 (0.75)
2,2,1.47 (0.70),159256.86 (36755.95),30.52 (2.94),9.24 (1.89),24.70 (3.75),23.90 (3.43),67.72 (3.62),15.35 (2.43),11.98 (2.05),15.23 (4.32),67.00 (24.81),4.61 (4.26),3.29 (0.79)
3,3,1.67 (0.64),216500.00 (53518.69),26.41 (1.50),13.21 (1.98),31.42 (3.36),18.77 (1.77),68.45 (3.94),15.72 (2.53),9.57 (1.98),15.20 (2.36),74.57 (18.10),5.02 (3.26),3.40 (0.31)
4,4,1.20 (0.72),148779.06 (36570.84),31.91 (3.20),8.70 (1.25),25.04 (1.49),24.65 (1.96),67.77 (3.09),14.19 (2.61),12.56 (1.57),15.77 (5.73),52.07 (29.94),3.29 (3.62),3.64 (0.68)
5,5,0.62 (nan),184750.00 (nan),30.20 (nan),7.20 (nan),22.70 (nan),24.29 (nan),68.45 (nan),18.35 (nan),12.21 (nan),20.87 (nan),45.39 (nan),0.86 (nan),3.84 (nan)
6,6,1.40 (0.59),175181.31 (45475.29),31.51 (2.47),10.22 (2.67),26.77 (4.61),22.76 (3.23),67.50 (2.94),15.64 (1.13),11.57 (2.26),15.63 (3.91),67.46 (15.40),3.73 (2.94),3.26 (0.75)
7,7,1.77 (0.69),151326.05 (46328.89),31.04 (3.07),9.08 (2.25),24.18 (4.65),24.33 (3.80),66.91 (4.17),16.06 (2.90),12.37 (4.09),14.32 (3.69),76.31 (28.74),6.13 (4.33),3.19 (0.74)
8,8,0.94 (0.63),169763.54 (29175.24),32.01 (3.72),8.67 (1.72),25.89 (4.08),24.03 (3.31),68.43 (3.00),16.34 (2.61),11.43 (1.57),19.76 (5.07),47.17 (30.81),2.18 (2.44),3.84 (0.51)
9,9,0.89 (0.29),220000.00 (30405.59),34.70 (0.05),8.69 (0.66),27.86 (0.11),20.62 (2.38),69.42 (2.54),15.18 (4.12),9.92 (0.44),23.30 (5.95),52.87 (15.21),1.48 (0.71),3.81 (0.27)


In [6]:
ball_sizes = (
    ball_census_stats
    .groupby("ball", as_index=False)["n_ttwas"]
    .first()
    .rename(columns={"n_ttwas": "n"})
)

mean_sd_wide = ball_sizes.merge(
    mean_sd_wide,
    on="ball",
    how="left"
)

mean_sd_wide

,ball,n,log_pop_density,median_house_price,pct_econ_inactive,pct_higher_managerial,pct_level4_plus,pct_no_quals,pct_owned,pct_private_rented,pct_routine,pct_self_employed,pct_urban,pop_density,urban_rural_score
0,0,70,1.22 (0.73),150090.14 (31379.76),31.80 (3.26),8.25 (1.64),23.55 (3.59),25.07 (3.53),67.54 (3.38),15.38 (2.48),12.59 (2.13),15.87 (5.05),58.98 (26.68),3.54 (4.14),3.45 (0.81)
1,1,9,1.29 (0.55),177105.61 (51028.48),31.17 (5.58),10.57 (3.48),26.13 (5.37),23.29 (5.47),68.31 (4.27),14.74 (2.09),11.69 (2.70),15.31 (4.21),62.55 (19.09),3.22 (2.87),3.37 (0.75)
2,2,84,1.47 (0.70),159256.86 (36755.95),30.52 (2.94),9.24 (1.89),24.70 (3.75),23.90 (3.43),67.72 (3.62),15.35 (2.43),11.98 (2.05),15.23 (4.32),67.00 (24.81),4.61 (4.26),3.29 (0.79)
3,3,3,1.67 (0.64),216500.00 (53518.69),26.41 (1.50),13.21 (1.98),31.42 (3.36),18.77 (1.77),68.45 (3.94),15.72 (2.53),9.57 (1.98),15.20 (2.36),74.57 (18.10),5.02 (3.26),3.40 (0.31)
4,4,9,1.20 (0.72),148779.06 (36570.84),31.91 (3.20),8.70 (1.25),25.04 (1.49),24.65 (1.96),67.77 (3.09),14.19 (2.61),12.56 (1.57),15.77 (5.73),52.07 (29.94),3.29 (3.62),3.64 (0.68)
5,5,1,0.62 (nan),184750.00 (nan),30.20 (nan),7.20 (nan),22.70 (nan),24.29 (nan),68.45 (nan),18.35 (nan),12.21 (nan),20.87 (nan),45.39 (nan),0.86 (nan),3.84 (nan)
6,6,8,1.40 (0.59),175181.31 (45475.29),31.51 (2.47),10.22 (2.67),26.77 (4.61),22.76 (3.23),67.50 (2.94),15.64 (1.13),11.57 (2.26),15.63 (3.91),67.46 (15.40),3.73 (2.94),3.26 (0.75)
7,7,11,1.77 (0.69),151326.05 (46328.89),31.04 (3.07),9.08 (2.25),24.18 (4.65),24.33 (3.80),66.91 (4.17),16.06 (2.90),12.37 (4.09),14.32 (3.69),76.31 (28.74),6.13 (4.33),3.19 (0.74)
8,8,23,0.94 (0.63),169763.54 (29175.24),32.01 (3.72),8.67 (1.72),25.89 (4.08),24.03 (3.31),68.43 (3.00),16.34 (2.61),11.43 (1.57),19.76 (5.07),47.17 (30.81),2.18 (2.44),3.84 (0.51)
9,9,2,0.89 (0.29),220000.00 (30405.59),34.70 (0.05),8.69 (0.66),27.86 (0.11),20.62 (2.38),69.42 (2.54),15.18 (4.12),9.92 (0.44),23.30 (5.95),52.87 (15.21),1.48 (0.71),3.81 (0.27)


In [7]:
mean_sd_wide = mean_sd_wide.merge(
    profiles[
        ["ball", "component_status", "is_isolate"]
    ].drop_duplicates(subset="ball"),
    on="ball",
    how="left",
    validate="one_to_one"
)

# Put identifying columns first
mean_sd_wide = mean_sd_wide[
    [
        "ball",
        "component_status",
        "is_isolate",
        "n",
        *[
            col for col in mean_sd_wide.columns
            if col not in {
                "ball",
                "component_status",
                "is_isolate",
                "n"
            }
        ]
    ]
]

mean_sd_wide

,ball,component_status,is_isolate,n,log_pop_density,median_house_price,pct_econ_inactive,pct_higher_managerial,pct_level4_plus,pct_no_quals,pct_owned,pct_private_rented,pct_routine,pct_self_employed,pct_urban,pop_density,urban_rural_score
0,0,Principal component,False,70,1.22 (0.73),150090.14 (31379.76),31.80 (3.26),8.25 (1.64),23.55 (3.59),25.07 (3.53),67.54 (3.38),15.38 (2.48),12.59 (2.13),15.87 (5.05),58.98 (26.68),3.54 (4.14),3.45 (0.81)
1,1,Principal component,False,9,1.29 (0.55),177105.61 (51028.48),31.17 (5.58),10.57 (3.48),26.13 (5.37),23.29 (5.47),68.31 (4.27),14.74 (2.09),11.69 (2.70),15.31 (4.21),62.55 (19.09),3.22 (2.87),3.37 (0.75)
2,2,Principal component,False,84,1.47 (0.70),159256.86 (36755.95),30.52 (2.94),9.24 (1.89),24.70 (3.75),23.90 (3.43),67.72 (3.62),15.35 (2.43),11.98 (2.05),15.23 (4.32),67.00 (24.81),4.61 (4.26),3.29 (0.79)
3,3,Principal component,False,3,1.67 (0.64),216500.00 (53518.69),26.41 (1.50),13.21 (1.98),31.42 (3.36),18.77 (1.77),68.45 (3.94),15.72 (2.53),9.57 (1.98),15.20 (2.36),74.57 (18.10),5.02 (3.26),3.40 (0.31)
4,4,Principal component,False,9,1.20 (0.72),148779.06 (36570.84),31.91 (3.20),8.70 (1.25),25.04 (1.49),24.65 (1.96),67.77 (3.09),14.19 (2.61),12.56 (1.57),15.77 (5.73),52.07 (29.94),3.29 (3.62),3.64 (0.68)
5,5,Disconnected,True,1,0.62 (nan),184750.00 (nan),30.20 (nan),7.20 (nan),22.70 (nan),24.29 (nan),68.45 (nan),18.35 (nan),12.21 (nan),20.87 (nan),45.39 (nan),0.86 (nan),3.84 (nan)
6,6,Principal component,False,8,1.40 (0.59),175181.31 (45475.29),31.51 (2.47),10.22 (2.67),26.77 (4.61),22.76 (3.23),67.50 (2.94),15.64 (1.13),11.57 (2.26),15.63 (3.91),67.46 (15.40),3.73 (2.94),3.26 (0.75)
7,7,Principal component,False,11,1.77 (0.69),151326.05 (46328.89),31.04 (3.07),9.08 (2.25),24.18 (4.65),24.33 (3.80),66.91 (4.17),16.06 (2.90),12.37 (4.09),14.32 (3.69),76.31 (28.74),6.13 (4.33),3.19 (0.74)
8,8,Principal component,False,23,0.94 (0.63),169763.54 (29175.24),32.01 (3.72),8.67 (1.72),25.89 (4.08),24.03 (3.31),68.43 (3.00),16.34 (2.61),11.43 (1.57),19.76 (5.07),47.17 (30.81),2.18 (2.44),3.84 (0.51)
9,9,Disconnected,True,2,0.89 (0.29),220000.00 (30405.59),34.70 (0.05),8.69 (0.66),27.86 (0.11),20.62 (2.38),69.42 (2.54),15.18 (4.12),9.92 (0.44),23.30 (5.95),52.87 (15.21),1.48 (0.71),3.81 (0.27)


In [8]:
principal_no_quals = (
    mean_sd_wide.loc[
        mean_sd_wide["component_status"].eq("Principal component"),
        ["ball", "component_status", "n", "pct_econ_inactive"]
    ]
    .rename(columns={
        "pct_econ_inactive":
            "pct_econ_inactive_mean_sd"
    })
    .sort_values("ball")
    .reset_index(drop=True)
)

principal_no_quals

,ball,component_status,n,pct_econ_inactive_mean_sd
0,0,Principal component,70,31.80 (3.26)
1,1,Principal component,9,31.17 (5.58)
2,2,Principal component,84,30.52 (2.94)
3,3,Principal component,3,26.41 (1.50)
4,4,Principal component,9,31.91 (3.20)
5,6,Principal component,8,31.51 (2.47)
6,7,Principal component,11,31.04 (3.07)
7,8,Principal component,23,32.01 (3.72)
8,11,Principal component,83,31.19 (2.76)
9,12,Principal component,3,26.06 (1.25)


In [9]:
principal_4_quals = (
    mean_sd_wide.loc[
        mean_sd_wide["component_status"].eq("Principal component"),
        ["ball", "component_status", "n", "pct_level4_plus"]
    ]
    .rename(columns={
        "pct_level4_plus":
            "Highest_Qual_4_qualifications_mean_sd"
    })
    .sort_values("ball")
    .reset_index(drop=True)
)

principal_4_quals

,ball,component_status,n,Highest_Qual_4_qualifications_mean_sd
0,0,Principal component,70,23.55 (3.59)
1,1,Principal component,9,26.13 (5.37)
2,2,Principal component,84,24.70 (3.75)
3,3,Principal component,3,31.42 (3.36)
4,4,Principal component,9,25.04 (1.49)
5,6,Principal component,8,26.77 (4.61)
6,7,Principal component,11,24.18 (4.65)
7,8,Principal component,23,25.89 (4.08)
8,11,Principal component,83,23.38 (3.90)
9,12,Principal component,3,31.73 (2.94)


In [10]:
principal_routine = (
    mean_sd_wide.loc[
        mean_sd_wide["component_status"].eq("Principal component"),
        ["ball", "component_status", "n", "pct_urban", "pop_density", "urban_rural_score"]
    ]
    
    .sort_values("ball")
    .reset_index(drop=True)
)

principal_routine

,ball,component_status,n,pct_urban,pop_density,urban_rural_score
0,0,Principal component,70,58.98 (26.68),3.54 (4.14),3.45 (0.81)
1,1,Principal component,9,62.55 (19.09),3.22 (2.87),3.37 (0.75)
2,2,Principal component,84,67.00 (24.81),4.61 (4.26),3.29 (0.79)
3,3,Principal component,3,74.57 (18.10),5.02 (3.26),3.40 (0.31)
4,4,Principal component,9,52.07 (29.94),3.29 (3.62),3.64 (0.68)
5,6,Principal component,8,67.46 (15.40),3.73 (2.94),3.26 (0.75)
6,7,Principal component,11,76.31 (28.74),6.13 (4.33),3.19 (0.74)
7,8,Principal component,23,47.17 (30.81),2.18 (2.44),3.84 (0.51)
8,11,Principal component,83,61.96 (25.94),3.95 (4.26),3.38 (0.80)
9,12,Principal component,3,68.45 (26.16),3.15 (2.05),3.49 (0.42)


In [11]:
selected_balls = [12, 3, 17, 13, 11, 0, 14]

dominant_second_lq = (
    profiles.loc[
        profiles["ball"].isin(selected_balls),
        ["ball", "dominant_LQ", "second_LQ"]
    ]
    .assign(
        ball=lambda df: pd.Categorical(
            df["ball"],
            categories=selected_balls,
            ordered=True
        )
    )
    .sort_values("ball")
    .reset_index(drop=True)
)

dominant_second_lq

,ball,dominant_LQ,second_LQ
0,12,1.51,1.31
1,3,1.53,1.47
2,17,1.37,1.20
3,13,0.63,0.62
4,11,0.86,0.81
5,0,0.75,0.67
6,14,0.84,0.76


Selected Ball Table

In [12]:
selected_balls = [5, 9, 20, 21, 24, 25]

selected_ball_table = (
    profiles.loc[
        profiles["ball"].isin(selected_balls),
        [
            "ball",
            "n_ttwas",
            "dominant_sector",
            "dominant_LQ",
            "pct_no_quals_mean",
            "pct_level4_plus_mean",
            "median_house_price_mean",
        ]
    ]
    .assign(
        ball=lambda df: pd.Categorical(
            df["ball"],
            categories=selected_balls,
            ordered=True
        )
    )
    .sort_values("ball")
    .reset_index(drop=True)
    .rename(columns={
        "ball": "Ball",
        "n_ttwas": "n",
        "dominant_sector": "Dominant sector",
        "dominant_LQ": "Dominant LQ",
        "pct_no_quals_mean": "No qualifications (%)",
        "pct_level4_plus_mean": "Level 4+ (%)",
        "median_house_price_mean": "Median House Price (£)",
    })
)

selected_ball_table

,Ball,n,Dominant sector,Dominant LQ,No qualifications (%),Level 4+ (%),Median House Price (£)
0,5,1,Advertising,3.08,24.29,22.70,184750.0
1,9,2,Design,2.67,20.62,27.86,220000.0
2,20,1,Film_TV,2.64,18.05,37.19,270000.0
3,21,1,Film_TV,1.77,24.66,25.43,120000.0
4,24,1,Film_TV,9.93,22.65,26.83,212000.0
5,25,1,Publishing,15.20,22.91,23.96,148000.0


In [13]:
selected_ball_table["Dominant LQ"] = selected_ball_table["Dominant LQ"].round(2)
selected_ball_table["No qualifications (%)"] = selected_ball_table["No qualifications (%)"].round(1)
selected_ball_table["Level 4+ (%)"] = selected_ball_table["Level 4+ (%)"].round(1)
selected_ball_table["Median House Price (£)"] = selected_ball_table["Median House Price (£)"].round(0)

selected_ball_table

,Ball,n,Dominant sector,Dominant LQ,No qualifications (%),Level 4+ (%),Median House Price (£)
0,5,1,Advertising,3.08,24.3,22.7,184750.0
1,9,2,Design,2.67,20.6,27.9,220000.0
2,20,1,Film_TV,2.64,18.0,37.2,270000.0
3,21,1,Film_TV,1.77,24.7,25.4,120000.0
4,24,1,Film_TV,9.93,22.6,26.8,212000.0
5,25,1,Publishing,15.20,22.9,24.0,148000.0


In [14]:
selected_balls = [13, 11, 0, 14, 3, 12, 17, 7, 8, 18]

selected_ball_table = (
    profiles.loc[
        profiles["ball"].isin(selected_balls)
    ]
    .assign(
        ball=lambda df: pd.Categorical(
            df["ball"],
            categories=selected_balls,
            ordered=True
        ),
        dominant_sector_LQ=lambda df:
            df["dominant_sector"] + " (" + df["dominant_LQ"].round(2).astype(str) + ")"
    )
    .sort_values("ball")
    .loc[
        :,
        [
            "ball",
            "n_ttwas",
            "dominant_sector_LQ",
            "pct_no_quals_mean",
            "pct_self_employed_mean"
        ]
    ]
    .rename(columns={
        "ball": "Ball",
        "n_ttwas": "n",
        "dominant_sector_LQ": "Dominant sector (LQ)",
        "pct_no_quals_mean": "No qualifications (%)",
        "pct_self_employed_mean": "Self-employment (%)"
    })
    .reset_index(drop=True)
)

selected_ball_table["No qualifications (%)"] = (
    selected_ball_table["No qualifications (%)"].round(1)
)

selected_ball_table["Self-employment (%)"] = (
    selected_ball_table["Self-employment (%)"].round(1)
)

selected_ball_table

,Ball,n,Dominant sector (LQ),No qualifications (%),Self-employment (%)
0,13,78,Design (0.63),26.1,13.9
1,11,83,Architecture (0.86),25.1,15.4
2,0,70,Architecture (0.75),25.1,15.9
3,14,59,Design (0.84),24.8,14.8
4,3,3,IT_Software (1.53),18.8,15.2
5,12,3,Design (1.51),18.4,14.4
6,17,7,Architecture (1.37),19.9,16.4
7,7,11,Advertising (1.03),24.3,14.3
8,8,23,Music_Arts (1.31),24.0,19.8
9,18,12,Architecture (1.48),23.2,19.9


Investigating breadth versus depth:

In [15]:
lq_cols = [
    'Advertising_LQ',
    'Architecture_LQ',
    'Design_LQ',
    'Film_TV_LQ',
    'IT_Software_LQ',
    'Music_Arts_LQ',
    'Publishing_LQ'
]

In [16]:
profiles[
    ['ball'] + lq_cols
].head()

,ball,Advertising_LQ,Architecture_LQ,Design_LQ,Film_TV_LQ,IT_Software_LQ,Music_Arts_LQ,Publishing_LQ
0,0,0.34,0.75,0.67,0.21,0.44,0.61,0.49
1,1,0.48,0.88,1.07,0.30,0.72,0.69,2.01
2,2,0.45,0.80,0.86,0.30,0.66,0.70,0.36
3,3,1.02,1.17,1.47,0.89,1.53,1.08,0.50
4,4,0.29,0.75,0.83,0.94,0.45,0.82,0.30


In [17]:
# Strongest sectoral specialisation
profiles['max_LQ'] = profiles[lq_cols].max(axis=1)

# Average LQ across all seven sectors
profiles['mean_LQ'] = profiles[lq_cols].mean(axis=1)

# Number of sectors with LQ > 1
profiles['n_specialised'] = (
    profiles[lq_cols] > 1
).sum(axis=1)

In [18]:
profiles[
    [
        'ball',
        'component_status',
        'n_ttwas',
        'n_specialised',
        'max_LQ',
        'mean_LQ'
    ]
].sort_values('n_specialised')

,ball,component_status,n_ttwas,n_specialised,max_LQ,mean_LQ
0,0,Principal component,70,0,0.75,0.501429
13,13,Principal component,78,0,0.63,0.411429
11,11,Principal component,83,0,0.86,0.514286
4,4,Principal component,9,0,0.94,0.625714
14,14,Principal component,59,0,0.84,0.557143
2,2,Principal component,84,0,0.86,0.590000
7,7,Principal component,11,1,1.03,0.585714
28,28,Principal component,6,1,2.61,1.015714
5,5,Disconnected,1,2,3.08,0.968571
8,8,Principal component,23,2,1.31,0.708571


In [19]:
pd.crosstab(
    profiles['n_specialised'],
    profiles['component_status']
)

component_status,Disconnected,Principal component
n_specialised,,
0,0,6
1,0,2
2,4,3
3,5,2
4,1,1
5,1,2
6,2,0
7,1,0


In [20]:
profiles.groupby('component_status')['n_ttwas'].describe()

,count,mean,std,min,25%,50%,75%,max
component_status,,,,,,,,
Disconnected,14.0,1.071429,0.267261,1.0,1.00,1.0,1.00,2.0
Principal component,16.0,29.250000,32.509486,3.0,6.75,10.0,61.75,84.0


In [21]:
profiles.groupby('component_status')['n_specialised'].describe()

,count,mean,std,min,25%,50%,75%,max
component_status,,,,,,,,
Disconnected,14.0,3.642857,1.691933,2.0,2.25,3.0,4.75,7.0
Principal component,16.0,1.750000,1.807392,0.0,0.00,1.5,3.00,5.0


In [22]:
pd.crosstab(
    profiles['n_specialised'],
    profiles['component_status']
)

component_status,Disconnected,Principal component
n_specialised,,
0,0,6
1,0,2
2,4,3
3,5,2
4,1,1
5,1,2
6,2,0
7,1,0


In [23]:
profiles[
    ['ball', 'component_status', 'n_ttwas', 'n_specialised']
].sort_values(
    ['component_status', 'n_specialised']
)

,ball,component_status,n_ttwas,n_specialised
5,5,Disconnected,1,2
9,9,Disconnected,2,2
15,15,Disconnected,1,2
25,25,Disconnected,1,2
21,21,Disconnected,1,3
22,22,Disconnected,1,3
24,24,Disconnected,1,3
26,26,Disconnected,1,3
27,27,Disconnected,1,3
16,16,Disconnected,1,4


In [24]:
pc = profiles[
    profiles['component_status'] == 'Principal component'
].copy()

In [25]:
pc['n_specialised'].value_counts().sort_index()

n_specialised
0    6
1    2
2    3
3    2
4    1
5    2
Name: count, dtype: int64

In [26]:
pc[
    [
        'ball',
        'n_specialised',
        'max_LQ',
        'pct_level4_plus_mean'
    ]
].sort_values('n_specialised')

,ball,n_specialised,max_LQ,pct_level4_plus_mean
0,0,0,0.75,23.55
2,2,0,0.86,24.70
4,4,0,0.94,25.04
11,11,0,0.86,23.38
13,13,0,0.63,22.38
14,14,0,0.84,23.83
7,7,1,1.03,24.18
28,28,1,2.61,28.36
1,1,2,2.01,26.13
8,8,2,1.31,25.89


In [27]:
from scipy.stats import spearmanr, pearsonr

pearson_breadth = pearsonr(
    pc['n_specialised'],
    pc['pct_level4_plus_mean']
)

spearman_breadth = spearmanr(
    pc['n_specialised'],
    pc['pct_level4_plus_mean']
)

print("Pearson:", pearson_breadth)
print("Spearman:", spearman_breadth)

Pearson: PearsonRResult(statistic=np.float64(0.9200956634874019), pvalue=np.float64(4.505504756010007e-07))
Spearman: SignificanceResult(statistic=np.float64(0.9063783752792699), pvalue=np.float64(1.315778412009127e-06))


In [28]:
pearson_depth = pearsonr(
    pc['max_LQ'],
    pc['pct_level4_plus_mean']
)

spearman_depth = spearmanr(
    pc['max_LQ'],
    pc['pct_level4_plus_mean']
)

print("Pearson depth:", pearson_depth)
print("Spearman depth:", spearman_depth)

Pearson depth: PearsonRResult(statistic=np.float64(0.6443534286399979), pvalue=np.float64(0.007053821205201033))
Spearman depth: SignificanceResult(statistic=np.float64(0.8182489338103116), pvalue=np.float64(0.00010704340256592821))


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

X = pc[['n_specialised', 'max_LQ']]
y = pc['pct_level4_plus_mean']

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

print("R-squared:", r2_score(y, pred))
print("Intercept:", model.intercept_)
print("n_specialised coefficient:", model.coef_[0])
print("max_LQ coefficient:", model.coef_[1])

R-squared: 0.8744235586969398
Intercept: 22.753631929800036
n_specialised coefficient: 1.3035649520245267
max_LQ coefficient: 1.0180262283454107


Breadth Table using data from Nesta/Census over balls

In [30]:
import pandas as pd
import numpy as np

import statsmodels
print(statsmodels.__version__)

import statsmodels.api as sm

0.15.0


In [31]:
nesta = pd.read_csv('/users/scandimimi/Documents/Manchester/ERP/nesta_ball.csv', index_col=0)

In [32]:
census = pd.read_csv(
    "/Users/scandimimi/Documents/Manchester/ERP/census_filtered.csv"
)

In [33]:
nesta = nesta.reset_index()

nesta.head()

,ttwa,Region / Nation,All creative industries: Employment 2011-2014,not_creative: Employment 2011-2014,Advertising and marketing: Employment 2011-2014,Architecture: Employment 2011-2014,"Design: product, graphic and fashion design: Employment 2011-2014","Film, TV, video, radio and photography: Employment 2011-2014","IT, software and computer services: Employment 2011-2014","Music, performing and visual arts: Employment 2011-2014",...,"IT, software and computer services: Employment 2011-2014_share","Music, performing and visual arts: Employment 2011-2014_share",Publishing: Employment 2011-2014_share,Advertising and marketing: Employment 2011-2014_LQ,Architecture: Employment 2011-2014_LQ,"Design: product, graphic and fashion design: Employment 2011-2014_LQ","Film, TV, video, radio and photography: Employment 2011-2014_LQ","IT, software and computer services: Employment 2011-2014_LQ","Music, performing and visual arts: Employment 2011-2014_LQ",Publishing: Employment 2011-2014_LQ
0,Aberystwyth,Wales,376.00,19789.47368,30.50,29.50,8.25,21.50,96.25,63.25,...,0.004773,0.003137,0.006285,0.326870,0.728732,0.238555,0.191961,0.324471,0.878220,1.297171
1,Andover,South East,975.25,27864.28571,76.25,31.75,78.00,92.25,352.00,61.75,...,0.012205,0.002141,0.009822,0.571394,0.548416,1.577064,0.575920,0.829732,0.599515,2.026933
2,Ashford,South East,1467.75,41935.71429,103.75,67.75,84.00,74.50,901.75,141.75,...,0.020776,0.003266,0.002171,0.516592,0.777570,1.128490,0.309041,1.412359,0.914428,0.448141
3,Banbury,South East,1745.50,37945.65217,199.25,137.75,134.50,198.50,885.25,139.00,...,0.022303,0.003502,0.001291,1.084898,1.728830,1.975930,0.900432,1.516197,0.980555,0.266476
4,Bangor and Holyhead,Wales,866.50,37673.91304,12.00,55.75,30.25,269.00,305.25,151.00,...,0.007920,0.003918,0.001122,0.067290,0.720581,0.457670,1.256666,0.538422,1.097013,0.231594


In [34]:
[col for col in nesta.columns if 'LQ' in col or 'lq' in col]

['Advertising and marketing: Employment 2011-2014_LQ',
 'Architecture: Employment 2011-2014_LQ',
 'Design: product, graphic and fashion design: Employment 2011-2014_LQ',
 'Film, TV, video, radio and photography: Employment 2011-2014_LQ',
 'IT, software and computer services: Employment 2011-2014_LQ',
 'Music, performing and visual arts: Employment 2011-2014_LQ',
 'Publishing: Employment 2011-2014_LQ']

In [35]:
census = census.dropna(subset=["TTWA11NM"]).copy()

In [36]:
lq_cols = [
    col for col in nesta.columns
    if col.endswith("Employment 2011-2014_LQ")
]

print("Number of LQ columns:", len(lq_cols))
print(lq_cols)

Number of LQ columns: 7
['Advertising and marketing: Employment 2011-2014_LQ', 'Architecture: Employment 2011-2014_LQ', 'Design: product, graphic and fashion design: Employment 2011-2014_LQ', 'Film, TV, video, radio and photography: Employment 2011-2014_LQ', 'IT, software and computer services: Employment 2011-2014_LQ', 'Music, performing and visual arts: Employment 2011-2014_LQ', 'Publishing: Employment 2011-2014_LQ']


In [37]:
print("Census rows:", len(census))
print("Census unique TTWAs:", census["TTWA11NM"].nunique())

print("Nesta rows:", len(nesta))
print("Nesta unique TTWAs:", nesta["ttwa"].nunique())

Census rows: 173
Census unique TTWAs: 173
Nesta rows: 173
Nesta unique TTWAs: 173


In [38]:
census_clean = census.dropna(subset=["TTWA11NM"]).copy()

lq_cols = [
    col for col in nesta.columns
    if col.endswith("Employment 2011-2014_LQ")
]

print("LQ columns:", len(lq_cols))

df = census_clean.merge(
    nesta[["ttwa"] + lq_cols],
    left_on="TTWA11NM",
    right_on="ttwa",
    how="inner",
    validate="one_to_one"
)

print("Number of TTWAs:", len(df))
print("Unique TTWAs:", df["TTWA11NM"].nunique())

LQ columns: 7
Number of TTWAs: 173
Unique TTWAs: 173


In [39]:
df["breadth"] = (df[lq_cols] > 1).sum(axis=1)
df["depth"] = df[lq_cols].max(axis=1)

In [40]:
df[["TTWA11NM", "breadth", "depth"]].head()

,TTWA11NM,breadth,depth
0,Barnsley,0,0.947647
1,Bradford,0,0.770545
2,Halifax,2,1.136006
3,Skipton,2,2.787328
4,Dorchester and Weymouth,0,0.732758


In [41]:
complete_lq = df[lq_cols].notna().all(axis=1)

df["breadth"] = np.where(
    complete_lq,
    (df[lq_cols] > 1).sum(axis=1),
    np.nan
)

df["depth"] = np.where(
    complete_lq,
    df[lq_cols].max(axis=1),
    np.nan
)

print("TTWAs with complete LQs:", complete_lq.sum())
print("TTWAs excluded due to missing LQs:", (~complete_lq).sum())

TTWAs with complete LQs: 173
TTWAs excluded due to missing LQs: 0


In [42]:
import statsmodels.api as sm
import pandas as pd

outcomes = [
    "pct_level4_plus",
    "pct_no_quals",
    "pct_econ_inactive",
    "pct_self_employed",
    "pct_higher_managerial",
    "pct_routine"
]

results_list = []

for outcome in outcomes:

    model_df = df[
        [outcome, "breadth", "depth"]
    ].dropna()

    X = model_df[["breadth", "depth"]]
    X = sm.add_constant(X)

    y = model_df[outcome]

    model = sm.OLS(y, X).fit(cov_type="HC3")

    results_list.append({
        "outcome": outcome,
        "n": int(model.nobs),
        "R2": model.rsquared,
        "breadth_coef": model.params["breadth"],
        "breadth_p": model.pvalues["breadth"],
        "depth_coef": model.params["depth"],
        "depth_p": model.pvalues["depth"]
    })

results = pd.DataFrame(results_list)

results

,outcome,n,R2,breadth_coef,breadth_p,depth_coef,depth_p
0,pct_level4_plus,173,0.413039,2.269048,1.302428e-22,0.017166,0.961467
1,pct_no_quals,173,0.329878,-1.628546,1.371154e-14,-0.184373,0.570162
2,pct_econ_inactive,173,0.074682,-0.622720,4.039740e-04,-0.091217,0.728693
3,pct_self_employed,173,0.086420,0.897873,2.815194e-02,0.201617,0.794262
4,pct_higher_managerial,173,0.348523,1.080848,4.172246e-18,0.014249,0.930921
5,pct_routine,173,0.379554,-1.151712,5.112265e-12,-0.052494,0.857138


In [43]:
df["breadth"].value_counts().sort_index()

breadth
0.0    65
1.0    46
2.0    31
3.0    20
4.0     6
5.0     1
6.0     3
7.0     1
Name: count, dtype: int64

Table for ERP:

In [44]:
label_map = {
    'pct_level4_plus': 'Level 4+ qualifications',
    'pct_no_quals': 'No qualifications',
    'pct_econ_inactive': 'Economic inactivity',
    'pct_self_employed': 'Self-employment',
    'pct_higher_managerial': 'Higher managerial',
    'pct_routine': 'Routine occupations'
}

results_table = results.copy()

results_table['Outcome'] = results_table['outcome'].map(label_map)

results_table = results_table[
    [
        'Outcome',
        'n',
        'R2',
        'breadth_coef',
        'breadth_p',
        'depth_coef',
        'depth_p'
    ]
].copy()

results_table.columns = [
    'Outcome',
    'N',
    'R²',
    'Breadth coefficient',
    'Breadth p',
    'Depth coefficient',
    'Depth p'
]

results_table['R²'] = results_table['R²'].round(3)
results_table['Breadth coefficient'] = results_table['Breadth coefficient'].round(3)
results_table['Depth coefficient'] = results_table['Depth coefficient'].round(3)


def format_p(p):
    if p < 0.001:
        return '<.001'
    return f'{p:.3f}'.replace('0.', '.')

results_table['Breadth p'] = results_table['Breadth p'].apply(format_p)
results_table['Depth p'] = results_table['Depth p'].apply(format_p)

results_table

,Outcome,N,R²,Breadth coefficient,Breadth p,Depth coefficient,Depth p
0,Level 4+ qualifications,173,0.413,2.269,<.001,0.017,.961
1,No qualifications,173,0.330,-1.629,<.001,-0.184,.570
2,Economic inactivity,173,0.075,-0.623,<.001,-0.091,.729
3,Self-employment,173,0.086,0.898,.028,0.202,.794
4,Higher managerial,173,0.349,1.081,<.001,0.014,.931
5,Routine occupations,173,0.380,-1.152,<.001,-0.052,.857


In [46]:
import pandas as pd

lq_cols = [
    'Advertising and marketing: Employment 2011-2014_LQ',
    'Architecture: Employment 2011-2014_LQ',
    'Design: product, graphic and fashion design: Employment 2011-2014_LQ',
    'Film, TV, video, radio and photography: Employment 2011-2014_LQ',
    'IT, software and computer services: Employment 2011-2014_LQ',
    'Music, performing and visual arts: Employment 2011-2014_LQ',
    'Publishing: Employment 2011-2014_LQ',
]

cases = df.copy()
lqs = cases[lq_cols].apply(pd.to_numeric, errors="coerce")

# Exclude incomplete sector profiles rather than treating missing LQs as zero.
cases = cases.loc[lqs.notna().all(axis=1)].copy()
lqs = lqs.loc[cases.index]

cases["breadth"] = lqs.gt(1).sum(axis=1)
cases["depth"] = lqs.max(axis=1)
cases["highest_lq_sector"] = lqs.idxmax(axis=1)

# Differences from the unweighted mean across all TTWAs,
# expressed in percentage points.
cases["level4_vs_average_pp"] = (
    cases["pct_level4_plus"] - df["pct_level4_plus"].mean()
)
cases["no_quals_vs_average_pp"] = (
    cases["pct_no_quals"] - df["pct_no_quals"].mean()
)

candidates = (
    cases.loc[cases["breadth"].between(1, 2)]
    .sort_values("depth", ascending=False)
)

display(
    candidates[
        [
            "TTWA11NM",
            "breadth",
            "highest_lq_sector",
            "depth",
            "pct_level4_plus",
            "level4_vs_average_pp",
            "pct_no_quals",
            "no_quals_vs_average_pp",
        ]
    ].head(20).round(2)
)

,TTWA11NM,breadth,highest_lq_sector,depth,pct_level4_plus,level4_vs_average_pp,pct_no_quals,no_quals_vs_average_pp
14,Peterborough,2,Publishing: Employment 2011-2014_LQ,15.20,23.96,-1.10,22.91,-1.06
53,Colchester,2,Publishing: Employment 2011-2014_LQ,6.11,26.96,1.90,20.37,-3.60
122,Southampton,2,Publishing: Employment 2011-2014_LQ,3.12,29.46,4.40,18.83,-5.14
22,Barnstaple,2,Advertising and marketing: Employment 2011-201...,3.08,22.70,-2.36,24.29,0.32
3,Skipton,2,Architecture: Employment 2011-2014_LQ,2.79,31.58,6.52,20.41,-3.56
5,Falmouth,2,"Design: product, graphic and fashion design: E...",2.63,27.78,2.72,18.94,-5.03
23,Barrow-in-Furness,2,Publishing: Employment 2011-2014_LQ,2.54,22.64,-2.42,23.94,-0.03
105,Norwich,2,Publishing: Employment 2011-2014_LQ,2.40,25.11,0.05,23.80,-0.17
13,Penrith,2,"Design: product, graphic and fashion design: E...",2.37,27.02,1.96,22.91,-1.06
47,Canterbury,2,Architecture: Employment 2011-2014_LQ,2.30,27.48,2.42,22.07,-1.90


In [48]:
narrow_deep = (
    cases.loc[
        (cases["breadth"].between(1, 2)) &
        (cases["depth"] >= cases["depth"].median())
    ]
    .sort_values("depth", ascending=False)
)

broad = (
    cases.loc[cases["breadth"] >= 3]
    .sort_values(
        ["breadth", "pct_level4_plus"],
        ascending=[False, False]
    )
)

print("NARROW / DEEP CANDIDATES")
display(
    narrow_deep[
        [
            "TTWA11NM",
            "breadth",
            "highest_lq_sector",
            "depth",
            "pct_level4_plus",
            "level4_vs_average_pp",
            "pct_no_quals",
            "no_quals_vs_average_pp",
        ]
    ].head(15).round(2)
)

print("BROAD CANDIDATES")
display(
    broad[
        [
            "TTWA11NM",
            "breadth",
            "highest_lq_sector",
            "depth",
            "pct_level4_plus",
            "level4_vs_average_pp",
            "pct_no_quals",
            "no_quals_vs_average_pp",
        ]
    ].head(15).round(2)
)

NARROW / DEEP CANDIDATES


,TTWA11NM,breadth,highest_lq_sector,depth,pct_level4_plus,level4_vs_average_pp,pct_no_quals,no_quals_vs_average_pp
14,Peterborough,2,Publishing: Employment 2011-2014_LQ,15.20,23.96,-1.10,22.91,-1.06
53,Colchester,2,Publishing: Employment 2011-2014_LQ,6.11,26.96,1.90,20.37,-3.60
122,Southampton,2,Publishing: Employment 2011-2014_LQ,3.12,29.46,4.40,18.83,-5.14
22,Barnstaple,2,Advertising and marketing: Employment 2011-201...,3.08,22.70,-2.36,24.29,0.32
3,Skipton,2,Architecture: Employment 2011-2014_LQ,2.79,31.58,6.52,20.41,-3.56
5,Falmouth,2,"Design: product, graphic and fashion design: E...",2.63,27.78,2.72,18.94,-5.03
23,Barrow-in-Furness,2,Publishing: Employment 2011-2014_LQ,2.54,22.64,-2.42,23.94,-0.03
105,Norwich,2,Publishing: Employment 2011-2014_LQ,2.40,25.11,0.05,23.80,-0.17
13,Penrith,2,"Design: product, graphic and fashion design: E...",2.37,27.02,1.96,22.91,-1.06
47,Canterbury,2,Architecture: Employment 2011-2014_LQ,2.30,27.48,2.42,22.07,-1.90


BROAD CANDIDATES


,TTWA11NM,breadth,highest_lq_sector,depth,pct_level4_plus,level4_vs_average_pp,pct_no_quals,no_quals_vs_average_pp
91,London,7,"Film, TV, video, radio and photography: Employ...",2.64,37.19,12.13,18.05,-5.92
39,Brighton,6,"Music, performing and visual arts: Employment ...",3.11,35.46,10.40,17.42,-6.55
49,Cheltenham,6,"Design: product, graphic and fashion design: E...",1.92,35.00,9.94,17.46,-6.51
135,Tunbridge Wells,6,Publishing: Employment 2011-2014_LQ,2.75,34.14,9.08,17.24,-6.73
107,Oxford,5,Publishing: Employment 2011-2014_LQ,3.63,36.41,11.35,16.88,-7.09
71,Guildford and Aldershot,4,"IT, software and computer services: Employment...",3.97,36.04,10.98,15.72,-8.25
75,High Wycombe and Aylesbury,4,"IT, software and computer services: Employment...",2.04,35.10,10.04,16.73,-7.24
25,Bath,4,Publishing: Employment 2011-2014_LQ,2.00,34.37,9.31,17.12,-6.85
126,Stevenage and Welwyn Garden City,4,Advertising and marketing: Employment 2011-201...,1.29,29.30,4.24,18.94,-5.03
21,Banbury,4,"Design: product, graphic and fashion design: E...",1.98,28.51,3.45,19.86,-4.11


In [50]:
pairs = []

for _, narrow in narrow_deep.iterrows():
    for _, wide in broad.iterrows():

       
        if wide["depth"] >= narrow["depth"]:
            continue

        
        level4_difference = (
            wide["pct_level4_plus"] - narrow["pct_level4_plus"]
        )

        no_quals_difference = (
            narrow["pct_no_quals"] - wide["pct_no_quals"]
        )

        
        breadth_difference = (
            wide["breadth"] - narrow["breadth"]
        )

        pairs.append(
            {
                "narrow_ttwa": narrow["TTWA11NM"],
                "narrow_breadth": narrow["breadth"],
                "narrow_depth": narrow["depth"],
                "narrow_sector": narrow["highest_lq_sector"],
                "narrow_level4": narrow["pct_level4_plus"],
                "narrow_no_quals": narrow["pct_no_quals"],

                "broad_ttwa": wide["TTWA11NM"],
                "broad_breadth": wide["breadth"],
                "broad_depth": wide["depth"],
                "broad_sector": wide["highest_lq_sector"],
                "broad_level4": wide["pct_level4_plus"],
                "broad_no_quals": wide["pct_no_quals"],

                "breadth_gap": breadth_difference,
                "depth_advantage_narrow": (
                    narrow["depth"] - wide["depth"]
                ),
                "level4_advantage_broad": level4_difference,
                "no_quals_advantage_broad": no_quals_difference,
            }
        )

pairs = pd.DataFrame(pairs)


good_pairs = pairs.loc[
    (pairs["level4_advantage_broad"] > 0) &
    (pairs["no_quals_advantage_broad"] > 0)
].copy()


good_pairs["illustration_score"] = (
    good_pairs["breadth_gap"] +
    good_pairs["level4_advantage_broad"] +
    good_pairs["no_quals_advantage_broad"]
)

good_pairs = good_pairs.sort_values(
    "illustration_score",
    ascending=False
)

display(good_pairs.head(20).round(2))

,narrow_ttwa,narrow_breadth,narrow_depth,narrow_sector,narrow_level4,narrow_no_quals,broad_ttwa,broad_breadth,broad_depth,broad_sector,broad_level4,broad_no_quals,breadth_gap,depth_advantage_narrow,level4_advantage_broad,no_quals_advantage_broad,illustration_score
290,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Cheltenham,6,1.92,"Design: product, graphic and fashion design: E...",35.00,17.46,5,0.11,18.60,17.78,41.38
291,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Bath,4,2.00,Publishing: Employment 2011-2014_LQ,34.37,17.12,3,0.03,17.97,18.12,39.09
294,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Harrogate,3,1.87,Architecture: Employment 2011-2014_LQ,33.86,17.77,2,0.16,17.46,17.47,36.93
295,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Hexham,3,1.73,"Music, performing and visual arts: Employment ...",33.44,20.65,2,0.30,17.04,14.59,33.63
296,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Salisbury,3,1.66,"Music, performing and visual arts: Employment ...",30.90,18.69,2,0.38,14.50,16.55,33.05
292,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Stevenage and Welwyn Garden City,4,1.29,Advertising and marketing: Employment 2011-201...,29.30,18.94,3,0.74,12.90,16.30,32.20
297,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Bristol,3,1.12,Architecture: Employment 2011-2014_LQ,30.65,19.71,2,0.91,14.25,15.53,31.78
293,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Banbury,4,1.98,"Design: product, graphic and fashion design: E...",28.51,19.86,3,0.06,12.11,15.38,30.49
384,Bridlington,2,1.69,"Design: product, graphic and fashion design: E...",16.66,33.56,Salisbury,3,1.66,"Music, performing and visual arts: Employment ...",30.90,18.69,1,0.04,14.24,14.87,30.11
383,Bridlington,2,1.69,"Design: product, graphic and fashion design: E...",16.66,33.56,Stevenage and Welwyn Garden City,4,1.29,Advertising and marketing: Employment 2011-201...,29.30,18.94,2,0.41,12.64,14.62,29.26


In [51]:
best_per_narrow = (
    good_pairs
    .sort_values("illustration_score", ascending=False)
    .drop_duplicates(subset="narrow_ttwa")
    .head(20)
)

display(
    best_per_narrow[
        [
            "narrow_ttwa",
            "narrow_breadth",
            "narrow_depth",
            "narrow_sector",
            "narrow_level4",
            "narrow_no_quals",
            "broad_ttwa",
            "broad_breadth",
            "broad_depth",
            "broad_sector",
            "broad_level4",
            "broad_no_quals",
            "breadth_gap",
            "depth_advantage_narrow",
            "level4_advantage_broad",
            "no_quals_advantage_broad",
            "illustration_score",
        ]
    ].round(2)
)

,narrow_ttwa,narrow_breadth,narrow_depth,narrow_sector,narrow_level4,narrow_no_quals,broad_ttwa,broad_breadth,broad_depth,broad_sector,broad_level4,broad_no_quals,breadth_gap,depth_advantage_narrow,level4_advantage_broad,no_quals_advantage_broad,illustration_score
290,Skegness and Louth,1,2.03,Publishing: Employment 2011-2014_LQ,16.40,35.24,Cheltenham,6,1.92,"Design: product, graphic and fashion design: E...",35.00,17.46,5,0.11,18.60,17.78,41.38
384,Bridlington,2,1.69,"Design: product, graphic and fashion design: E...",16.66,33.56,Salisbury,3,1.66,"Music, performing and visual arts: Employment ...",30.90,18.69,1,0.04,14.24,14.87,30.11
254,Pembroke and Tenby,1,2.24,Architecture: Employment 2011-2014_LQ,22.52,26.12,Cheltenham,6,1.92,"Design: product, graphic and fashion design: E...",35.00,17.46,5,0.32,12.48,8.66,26.14
88,Barnstaple,2,3.08,Advertising and marketing: Employment 2011-201...,22.70,24.29,London,7,2.64,"Film, TV, video, radio and photography: Employ...",37.19,18.05,5,0.44,14.49,6.24,25.73
517,Lowestoft,1,1.25,"Design: product, graphic and fashion design: E...",18.64,29.61,Bristol,3,1.12,Architecture: Employment 2011-2014_LQ,30.65,19.71,2,0.13,12.01,9.90,23.91
0,Peterborough,2,15.20,Publishing: Employment 2011-2014_LQ,23.96,22.91,London,7,2.64,"Film, TV, video, radio and photography: Employ...",37.19,18.05,5,12.56,13.23,4.86,23.09
523,King's Lynn,1,1.18,Architecture: Employment 2011-2014_LQ,19.43,29.51,Bristol,3,1.12,Architecture: Employment 2011-2014_LQ,30.65,19.71,2,0.06,11.22,9.80,23.02
158,Barrow-in-Furness,2,2.54,Publishing: Employment 2011-2014_LQ,22.64,23.94,Cheltenham,6,1.92,"Design: product, graphic and fashion design: E...",35.00,17.46,4,0.62,12.36,6.48,22.84
460,Southend,1,1.42,"IT, software and computer services: Employment...",18.62,26.64,Stevenage and Welwyn Garden City,4,1.29,Advertising and marketing: Employment 2011-201...,29.30,18.94,3,0.13,10.68,7.70,21.38
456,Blackpool,1,1.43,Advertising and marketing: Employment 2011-201...,20.26,28.07,Stevenage and Welwyn Garden City,4,1.29,Advertising and marketing: Employment 2011-201...,29.30,18.94,3,0.14,9.04,9.13,21.17


In [53]:
import statsmodels.api as sm
import pandas as pd

outcomes = [
    'pct_level4_plus',
    'pct_no_quals',
    'pct_econ_inactive',
    'pct_self_employed',
    'pct_higher_managerial',
    'pct_routine'
]

results_list = []

for outcome in outcomes:

    model_df = df[[outcome, 'breadth', 'depth']].dropna()

    X = model_df[['breadth', 'depth']]
    X = sm.add_constant(X)

    y = model_df[outcome]

    model = sm.OLS(y, X).fit(cov_type='HC3')

    results_list.append({
        'outcome': outcome,
        'n': int(model.nobs),
        'R2': model.rsquared,
        'breadth_coef': model.params['breadth'],
        'breadth_p': model.pvalues['breadth'],
        'depth_coef': model.params['depth'],
        'depth_p': model.pvalues['depth']
    })

results = pd.DataFrame(results_list)

results

,outcome,n,R2,breadth_coef,breadth_p,depth_coef,depth_p
0,pct_level4_plus,173,0.413039,2.269048,1.302428e-22,0.017166,0.961467
1,pct_no_quals,173,0.329878,-1.628546,1.371154e-14,-0.184373,0.570162
2,pct_econ_inactive,173,0.074682,-0.622720,4.039740e-04,-0.091217,0.728693
3,pct_self_employed,173,0.086420,0.897873,2.815194e-02,0.201617,0.794262
4,pct_higher_managerial,173,0.348523,1.080848,4.172246e-18,0.014249,0.930921
5,pct_routine,173,0.379554,-1.151712,5.112265e-12,-0.052494,0.857138
